In [1]:
from pathlib import Path

import util
from util import workflow

browser = False
file = util.notebook_file() if util.is_notebook() else __file__
tag = util.file_tag(file)
root_path = Path("..")

In [2]:
# Build
from automol.graph import enum

import automech
from automech.species import Species

par_mech = workflow.read_parent_mechanism(root_path=root_path)
mech = automech.from_smiles(
    spc_smis=["C12C(O2)[CH]CC1", "C12C(O2)C[CH]C1"],
    src_mech=par_mech,
)
#  - O2 additions
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.o2_addition,
    rcts_=[None, "[O][O]"],
    spc_col_=Species.smiles,
    src_mech=par_mech,
)
#  - HO2 elimination
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.ho2_elimination,
    spc_col_=Species.smiles,
    src_mech=par_mech,
)
#  - drop alpha HO2 elimination that we can't currently handle
mech = automech.drop_reactions_by_smiles(
    # >> suspiciously low frequency, could not be confirmed by IRC due to failure
    mech,
    rxn_smis=["[O]OC1CCC2C1O2>>[O]O.C=1CCC2C=1O2"],
)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
# Write
workflow.write(mech=mech, tag=tag, root_path=root_path, browser=browser)


Finalizing build for...
reactions=shape: (4, 5)
┌────────────────────────┬───────────────────────┬───────────┬────────────┬────────────────────────┐
│ reactants              ┆ products              ┆ formula   ┆ reversible ┆ rate_constant          │
│ ---                    ┆ ---                   ┆ ---       ┆ ---        ┆ ---                    │
│ list[str]              ┆ list[str]             ┆ struct[3] ┆ bool       ┆ struct[17]             │
╞════════════════════════╪═══════════════════════╪═══════════╪════════════╪════════════════════════╡
│ ["S(1690)"]            ┆ ["HO2(8)", "S(1288)"] ┆ {5,7,3}   ┆ true       ┆ {1,{null,null,null,nul │
│                        ┆                       ┆           ┆            ┆ l,null,n…              │
│ ["S(1691)"]            ┆ ["HO2(8)", "S(1288)"] ┆ {5,7,3}   ┆ true       ┆ {1,{null,null,null,nul │
│                        ┆                       ┆           ┆            ┆ l,null,n…              │
│ ["O2(6)", "S(1289)"]   ┆ ["S(1691)"]    

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]


Writing mechanism...
../data/D_r-o2_p2v0_orig.json
../data/D_r-o2_p2v0.json
../data/mechanalyzer/D_r-o2_p2v0.dat
../data/mechanalyzer/D_r-o2_p2v0.csv


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]


Stereoexpansion errors:


In [4]:
# # Read
# workflow.read(tag=tag, root_path=root_path)

In [5]:
# # Simulate
# workflow.simulate(full_tag=f"full_{tag}_calc", root_path=root_path)
# workflow.simulate(full_tag=f"full_{tag}_control", root_path=root_path)

In [6]:
# # Plot
# charts = workflow.plot(
#     full_tag=f"full_{tag}_calc",
#     x_col="O2_molecules",
#     y_col_=["C5H8(522)", "C5H8O(825)rs"],
#     root_path=root_path,
#     line_source_=["hill", "lokachari"],
#     point_source="experiment",
# )
# for chart in charts:
#     chart.show()

In [ ]:
# import automech
# from mechdriver.subtasks import display

# chan = "1: 2"

# # TRANSITION STATE
# #   - Read in expanded mechanism
# mech_path = util.p_.mechanism(tag, ext="json", path=util.p_.data(root_path))
# mech = automech.io.read(mech_path)

# #   - Display the reaction
# automech.display_reactions(mech, chans=[chan])

In [8]:
# #   - Display the TS mode
# calc_path = util.p_.calc(root_path, tag)
# display("conf_opt", chan, path=calc_path)

In [ ]:
# #   - Display the TS mode
# calc_path = util.p_.calc(root_path, tag)
# display("find_ts", chan, path=calc_path)

In [10]:
# #   - Display the IRC
# calc_path = util.p_.calc(root_path, tag)
# display("rpath_scan", chan, path=calc_path)

In [11]:
# # REACTION RATE
# #   - Read in calculated mechanism
# cal_mech = automech.io.read(data_path / f"{tag}_calc.json")

# #   - Read in other mechanisms for comparison
# par_mech = automech.io.read(data_path / "full_raw.json")
# tags0 = util.previous_tags(tag)
# trues = [True] * len(tags0)
# names0 = list(map(util.calculated_mechanism_name, tags0))
# mechs0 = [automech.io.read(data_path / f"{name}.json") for name in names0]

# #   - Display the reaction and calculated rate
# automech.display_reactions(
#     cal_mech,
#     chans=[chan],
#     comp_mechs=[par_mech, *mechs0],
#     comp_labels=["Hill", *tags0],
#     comp_stereo=[False, *trues],
# )